# 3D nnU-Net brain tumour segmentation (Colab)

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/brain-tumor-nnunet'    
DATASET_ID = 501
TRAINER = 'nnUNetTrainer_100epochs'   # 100 epochs; use nnUNetTrainer_50epochs for a faster run, nnUNetTrainer for the full 1000
FOLD = '0'
RESUME = False                        # set True to continue after a disconnect

os.environ['nnUNet_raw'] = '/content/nnunet/raw'
os.environ['nnUNet_preprocessed'] = '/content/nnunet/preprocessed'
os.environ['nnUNet_results'] = '/content/drive/MyDrive/nnunet_results'   # on Drive so checkpoints persist
for k in ('nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results'):
    os.makedirs(os.environ[k], exist_ok=True)
assert os.path.isdir(PROJECT), 'Project folder not found on Drive'
R = f"{os.environ['nnUNet_raw']}/Dataset{DATASET_ID}_BrainTumour"

In [ ]:
!pip -q install nnunetv2 nibabel

### Restore from cache
Requires the cache cell below to have been run once.

In [ ]:
CACHE = '/content/drive/MyDrive/nnunet_cache'
PRE = os.environ['nnUNet_preprocessed']; RAW = os.environ['nnUNet_raw']
!tar -xf {CACHE}/preprocessed.tar -C {PRE}
!mkdir -p {RAW}/Dataset{DATASET_ID}_BrainTumour && tar -xf {CACHE}/raw_test.tar -C {RAW}/Dataset{DATASET_ID}_BrainTumour

## 1. Get the data (Medical Segmentation Decathlon, Task01_BrainTumour, CC-BY-SA 4.0)
If this download fails, download `Task01_BrainTumour.tar` from http://medicaldecathlon.com, put it in `/content`, and re-run the extract cell.

In [ ]:
!test -f /content/Task01_BrainTumour.tar || wget -q --show-progress -O /content/Task01_BrainTumour.tar https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar
!test -d /content/msd/Task01_BrainTumour || (mkdir -p /content/msd && tar -xf /content/Task01_BrainTumour.tar -C /content/msd)
!ls /content/msd/Task01_BrainTumour

## 2. Convert to nnU-Net format (holds out 10% of cases as a test set) and preprocess

In [ ]:
!test -d {R} || python {PROJECT}/scripts/prepare_dataset.py --msd-dir /content/msd/Task01_BrainTumour --dataset-id {DATASET_ID}

In [ ]:
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 3d_fullres --verify_dataset_integrity

### Cache preprocessed data to Drive (run once, after preprocessing)

In [ ]:
CACHE = '/content/drive/MyDrive/nnunet_cache'
PRE = os.environ['nnUNet_preprocessed']; RAW = os.environ['nnUNet_raw']
!mkdir -p {CACHE}
!tar -cf {CACHE}/preprocessed.tar -C {PRE} Dataset{DATASET_ID}_BrainTumour
!tar -cf {CACHE}/raw_test.tar -C {RAW}/Dataset{DATASET_ID}_BrainTumour imagesTs labelsTs dataset.json split.json
!ls -lh {CACHE}

## 3. Train
Look at the time per epoch printed in the log, then decide whether to stop early and use the latest checkpoint.
nnU-Net saves `checkpoint_latest.pth` periodically and `checkpoint_best.pth`, in your Drive results folder.

In [ ]:
cont = '--c' if RESUME else ''
!nnUNetv2_train {DATASET_ID} 3d_fullres {FOLD} -tr {TRAINER} {cont} --npz

## 4. Predict on the held-out test set and evaluate

In [ ]:
!nnUNetv2_predict -i {R}/imagesTs -o /content/preds_test -d {DATASET_ID} -c 3d_fullres -tr {TRAINER} -f {FOLD} -chk checkpoint_best.pth
!cd {PROJECT}/scripts && python evaluate.py --pred /content/preds_test --gt {R}/labelsTs --images {R}/imagesTs --out {PROJECT}/results

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob(f'{PROJECT}/results/figures/*.png'))[:4]:
    display(Image(f))
display(Image(f'{PROJECT}/results/dice_boxplot.png'))
!cat {PROJECT}/results/summary.json | head -40